# Cassava Leaf Disease — EfficientNet-B0 Inference (v3 + 4-way TTA)

**Before running**
1. Add Input → competition `cassava-leaf-disease-classification`
2. Add Input → Dataset with latest `efficientnet_b0.pt` (v3)
3. GPU on, Internet Off
4. Save Version → Save & Run All → Submit

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b0

IMG_SIZE = 384
BATCH = 32
NUM_CLASSES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT = Path("/kaggle/input")
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def find_comp_root() -> Path:
    candidates = [
        INPUT / "cassava-leaf-disease-classification",
        INPUT / "competitions" / "cassava-leaf-disease-classification",
    ]
    for root in candidates:
        if (root / "sample_submission.csv").exists() or (root / "test_images").exists():
            return root
    samples = list(INPUT.rglob("sample_submission.csv"))
    if samples:
        return samples[0].parent
    raise FileNotFoundError("Add competition data to this notebook.")


COMP = find_comp_root()
TEST_DIR = COMP / "test_images"
SAMPLE = COMP / "sample_submission.csv"
weight_files = (
    list(INPUT.rglob("efficientnet_b0_v3.pt"))
    + list(INPUT.rglob("efficientnet_b0.pt"))
)
WEIGHTS = weight_files[0]
print("device", DEVICE)
print("comp", COMP)
print("weights", WEIGHTS)
print("test_dir", TEST_DIR, "exists", TEST_DIR.exists())

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image = Image.open(self.image_dir / image_id).convert("RGB")
        return self.transform(image), image_id


def make_transform(hflip=False, vflip=False):
    ops = [transforms.Resize(IMG_SIZE + 32), transforms.CenterCrop(IMG_SIZE)]
    if hflip:
        ops.append(transforms.RandomHorizontalFlip(p=1.0))
    if vflip:
        ops.append(transforms.RandomVerticalFlip(p=1.0))
    ops.extend([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return transforms.Compose(ops)


def build_model():
    model = efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    state = torch.load(WEIGHTS, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


model = build_model()
print("loaded")

In [ ]:
if TEST_DIR.exists():
    test_ids = sorted(p.name for p in TEST_DIR.glob("*.jpg"))
else:
    test_ids = []
if not test_ids:
    test_ids = pd.read_csv(SAMPLE)["image_id"].tolist()

tta = [(False, False), (True, False), (False, True), (True, True)]
probs = np.zeros((len(test_ids), NUM_CLASSES), dtype=np.float64)
for hflip, vflip in tta:
    loader = DataLoader(
        LeafDataset(test_ids, TEST_DIR, make_transform(hflip, vflip)),
        batch_size=BATCH,
        shuffle=False,
        num_workers=2,
        pin_memory=DEVICE.type == "cuda",
    )
    offset = 0
    with torch.no_grad():
        for images, image_ids in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            batch = torch.softmax(logits.float(), dim=1).cpu().numpy()
            probs[offset : offset + len(image_ids)] += batch
            offset += len(image_ids)
probs /= float(len(tta))

submission = pd.DataFrame({"image_id": test_ids, "label": probs.argmax(1).astype(int)})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("rows", len(submission))
print(submission["label"].value_counts().sort_index())